# כמיהה в метаязыке NSM: консистентность языковых моделей при промптинге на русском, английском и иврите

Ноутбук воспроизводит эксперимент целиком:
1. запросы к моделям через OpenRouter API (2 модели × 3 языка × 5 итераций = 30 толкований);
2. выгрузка ответов в таблицу для слепой экспертной оценки;
3. автоматическая проверка критерия К1 (только примитивы);
4. подсчёт среднего и стандартного отклонения по экспертным баллам.

Перед запуском: Среда выполнения → Выполнить по очереди сверху вниз. Нужен только ключ OpenRouter (бесплатные модели, баланс не расходуется).

In [ ]:
import requests, json, time, re, random, os
import pandas as pd
from getpass import getpass

## 1. Ключ OpenRouter
Ключ вводится в скрытое поле и не сохраняется в ноутбуке.

In [ ]:
OPENROUTER_KEY = getpass('OpenRouter API key: ').strip()

## 2. Параметры эксперимента
Одинаковые для всех моделей и языков. Используются только бесплатные версии моделей (`:free`).

In [ ]:
MODELS = [
    {"name": "Nemotron3", "id": "nvidia/nemotron-3-super-120b-a12b:free"},   # NVIDIA, США
    {"name": "Dots3",     "id": "dots-studio/dots-3-note-preview:free"},     # Xiaohongshu, Китай
]
LANGS       = ["ru", "en", "he"]
N_ITER      = 5
TEMPERATURE = 0.7
MAX_TOKENS  = 16000   # с запасом: модели «рассуждают» перед ответом
PAUSE       = 5       # с между запросами (лимит бесплатных моделей — 20 запросов в минуту)
TRIES       = 2       # пустой ответ или ошибка → один повтор
OUT         = "results.json"

## 3. Промпты
Три версии одного промпта; языковые версии — переводы друг друга. 24 корпусных контекста во всех версиях даны на иврите. Те же тексты лежат в файлах `prompt_ru.txt`, `prompt_en.txt`, `prompt_he.txt`.

In [ ]:
PROMPTS = {
 "ru": "Ты — специалист по лексической семантике, работающий в рамках теории естественного семантического метаязыка (Natural Semantic Metalanguage, NSM) А. Вежбицкой и К. Годдарда.\n\nЗАДАЧА\nСоставь семантическое толкование ивритского существительного כמיהה на естественном семантическом метаязыке.\n\nМАТЕРИАЛ\nНиже даны 24 реальных употребления этого слова из корпуса современного иврита. Внимательно изучи их: толкование должно описывать значение, общее для всех этих употреблений.\n\n1. הרחבה. למעשה, אם לא מאחורי כל פנייה בין אישית, לפחות מאחורי כל פנייה לטיפול מסתתרים צרכים ראשוניים, מסתתרת כמיהה ראשונית להיענות אימהית, ותפקידו של המטפל הוא לזהות אותה. עדיין היענות לצרכים אלה אינה פשוטה והיא דורשת סוג של אומנות,\n2. מספור, דרך מאות שנים של סבל וגלות, מעשי טבח ואף רצח עם. אך למרות כל זאת, העם היהודי שמר על זהותו הייחודית ועל מסורותיו וכן על כמיהתו הארוכה לשוב למולדת.' כך דיבר אובמה על זכות הישראלים על אדמת פלסטין... אתם מתארים לעצמכם את הנשיא האמריקאי מקריא את הקטע ה\n3. סופת טורנדו. בעקבות זעמה מתעוררת בדורותי המשאלה הילדית לעזוב את ההורים \"הרעים\" ולהתחיל מחדש. הביקורת על ההורים, כמו הכמיהה לעזוב אותם, אינה מודעת אצל ילדים צעירים. החשיבה הביקורתית נתפסת כבוגדנית ומעוררת אשמה, בושה וחרדה ולכן היא מודחקת אל מ\n4. של מדינת ישראל, אלא בקוממיותו של האזרח הערבי הנרמס על ידי מדינת הלאום המדירה, הדכאנית והמושחתת שלו. אולם יש במאמר געגוע, כמיהה, למילה שרבים בשמאל הרדיקלי (בארץ ובחו\"ל) שכחו, או היו מעדיפים לחיות בלעדיו: מדינה. אלה שחיים בתוך מדינה חזקה, יציבה,\n5. והנשימות שלי מואצות. האם אי-פעם יגיע הרגע שבו אהיה לידו בלי להשתוקק אליו? בלי לייחל למגע עורו כנגד עורי? זאת אפילו לא כמיהה מינית – לפחות לא לגמרי. זה צורך. כאילו ההישרדות שלי תלויה בו. כאילו אנחנו שני חצאים של שלם אחד ואף אחד מאיתנו לא יכול לשרוד\n6. שסקס הוא הכוח החזק ביותר שמאפשר את מערכת היחסים. הם טועים. אהבה רומנטית, להבדיל מערנות מינית, היא למעשה חזקה יותר. הכמיהה לאהבה רומנטית יכולה להיות חזקה יותר מהרצון לחיות, וחזקה הרבה יותר מכל דחף מיני שהוא. הלן פישר, אנתרופולוגית אמריקאית,\n7. שיטת סאטיה כל אדם שואף להרמוניה, רווחה אישית ושקט נפשי. רצונו של האדם על מכלול מטרותיו, שאיפותיו וחלומותיו, מסתכם בכמיהתו לשקט, לתכלית וליושר פנימי המתקיימים בהלימה רגשית נפשית ופיסית בחייו. מה שנוכח כאן ועכשיו – התייחסותו של האדם להווה\n8. בדרכך ומספרת שאת לא צריכה לתת דין וחשבון כי לאף אחד מהצדדים אין באמת ציפיות. אז שימי לב איך מתעוררים אצלך רגשות נוספים כי הכמיהה לקרבה אמיתית לא מתכהה גם לנוכח תשוקה מינית עזה. תמיד תזכרי את הבוקר שאחרי וגם אם זה אומר שיש מקרים שתצאי מחדר השינה בלי\n9. וזלדה. ואלה רק שלושה. נסים אלוני אבֵל על מגע ידו המשחיר. שאול טשרניחובסקי אבֵל על אי-המגע. השושנה שלו הופכת לכמיהה בלתי מושגת ומתמדת, לתשוקה המאוכלת כבר למן ההתחלה. בדומה לאלוני, יש בפתיחת השיר מעין הבטחת-אושר שוות-כל, ברכת אל שורה מ\n10. . רוב הספרות הנוגעת בנושא, אם כן, רואה בחלומות שבהם מופיע המטפל מפורשות (ללא מסווה) כמשקפים בעיות קשות אצל המטופל )למשל, כמיהה עזה להתמזגות או פגיעה ביכולת הסימבולית) או אצל המטפל ובקשר הטיפולי )(למשל, קשיים הקשורים להעברה נגדית, חדירת המציאות,\n11. משנה תורה. במשנה תורה, לעומת זאת, פסגת אהבת ה', אשר יסודה היא בשבח ובהתפעמות מחכמתו וגדולתו המשתקפים בבריאה, היא צמאון וכמיהה להכרה נוספת. צמאון זה מתואר בהלכות תשובה (ראו בנספח; וגם קודם לכן שם, בהלכה ג) כשגיון וכאובססיה כאילו חולנית. כך, כנראה,\n12. עת שהמרחק רב בין הגופות מתנווט אני אלייך במחשבות כמדריך את התקוות מגעגועים אל משימות, מחלום אל הוויות, מציפיה אל עשייה, מכמיהה אל נגיעה, מאהבה אל תאווה. חכי לי עד אחזור ואלייך אחדור, כי נמאס מהחידלון שהמרחק מאלצו. במעגלי צדק תלפיות, יד חרוצים 19\n13. \", מולו. בתהליך זה תוך השתקפות באחר העומד מנגד מתפתח העצמי ובוקעת התודעה. (consciousness) בשלב מאוחר יותר בחיים כאשר הכמיהה להידבקות תהיה משוחררת מן הצורך הקמאי להשלים עצמי באמצעות החיבור, יוכל להתקיים סוף סוף החיבור והדבקות הפעם במישור ה\n14. קהילת יוצאי אתיופיה, חוגגים את חג הסיגד המרכז הרפואי ברזילי מברך את בני קהילת יוצאי אתיופיה, לרגל חג הסיגד. החג מסמל את הכמיהה לעלייה לארץ, ואת חידוש הברית של יהודי אתיופיה עם עמם ועם תורתם, והוא נחוג בכל שנה בכ\"ט בחשוון, 50 יום לאחר יום הכיפורים. ב\n15. של הצבא כגואל העם ומגן העצמאות, הם חונכו על מיתוס הפוך – הצבא כמרכז של כפירה, כמרד נגד השם, כמקום זימה ותועבה. אין לנו שום כמיהה להגיע למוסד הזה. שלישית – עזבנו מסגרת נוקשה עם כללים שאנו כופרים בהם. הצבא הוא מבחינות רבות מסגרת דומה. צריך להאמין ב\n16. באמת ובתמים צועדים בנתיבו של פטרוס. אלוהים שיבח את פטרוס על אמונת האמת ואהבת האמת שלו ועל נאמנותו לאלוהים. ובזכות כנותו וכמיהתו לאלוהים בלבו, אלוהים עשה אותו למושלם. אם אכן יש לכם אותה אהבה ואותה אמונה שהייתה לפטרוס, הרי שישוע לבטח יעשה אתכם ל\n17. , וניתן לאחד רוח ברוח. זוהי איננה כמיהה לבלתי מושג אלא כמיהה למושג. כאשר מתעוררת אצל האדם המשיכה המינית עליו לדעת שזוהי הכמיהה לבלתי מושג. אין לו אפשרות להגיע לרמת חיבור כזאת בעולם הזה. האשליה כאילו ... לחבר בין אהבה למשיכה מינית, בין מהות לצורה.\n18. שם ממשפחות חרדיות לעילא ולעילא, והוא רואה לעיתים תופעות מאכזבות, בהתחלה תמימותו נסדקת, ובהמשך הוא מושפע בעצמו... וה כמיהה הכנה שלו ל\"שבתי\" \"כל ימי חיי! \", ולדבקות מוחלטת בתורה ובנותנה, ולקנין גדלות של ממש ולירא\"ש ללא כחל ושרק, לפעמים, הבנין ה\n19. פרטני וזוגי בתחום הטיפול התומך בטיפולי פוריות, מסבירה על חווית ההתמודדות עם הטיפולים. \"הבה לי בנים, ואם אין מתה אנכי\": הכמיהה לילד והשפעתם של טיפולי הפוריות \"הבה לי בנים, ואם אין מתה אנכי\" (בראשית ל'), אומרת רחל המקראית ביאושה, אחרי שנים של עקרות.\n20. חלק מהשליחים והמגויסים טענו כי סמכות ההגנה אכן התקבלה בהתלהבות וללא עוררין. שמואל ארנר, שהתגייס להגנה בגרמניה, מתאר את כמיהתו ונכונות הנוער להצטרף לארגון ההגנה מתוך מטרה אחת: עלייה לארץ ישראל, ומתוך כך קיבלו רבים את סמכות ההגנה. ככל שעבר הזמן, ו\n21. בכך הזדמנות. הייתי אז חבר-כנסת. אינני אוהב, בלשון המעטה, את ההימנון הלאומי שלנו. המילים נכתבו לפני מאה שנה וביטאו את כמיהת היהודים בגולה לארץ-ישראל. גם אחרי ששונו המילים של הפזמון (''עוד לא אבדה תקוותנו / התקווה הנושנה / לשוב לארץ אבותינו / ה\n22. ולשכנע את האדם שהיא מרוקנת את חייו מכל משמעות והופכת אותם לחלולים. בשלב שני, צריך לעורר את הצד העליון שבאדם, להעיר את כמיהתו לרוחניות, לערכים, לאידיאלים, להציב בפניו אתגרים רוחניים, וממילא ישוב גם אל האמונה ואל הקדושה. לעומת זאת, את תרבות ה\n23. ערי רקיע ויגיעו עד לפני כיסא הכבוד. \"יותר גוף יותר כתיבה\" הלן סיסקו, \"צחוקה של המדוזה\" בשירה האמונית קיימת באופן אימננטי כמיהה וחתירה לקראת מפגש בלתי מתווך עם האל. בדברים שלהלן אבקש לבדוק דרכי התמודדות שונות במפגש עם האל, המוצעות בשיריהן של שלוש\n24. האלהיים אם לא יפותחו כראוי בחכמה האצילית ובחכמת החיים, בגבורת הנשמה ובגבורת הגוף, יכולים הם לפול ממדרגתם לידי איזה כמיהה חולנית, שינטל מהם כל זיום ותפארתם, עד אשר מה שיגרמו געגועים נחלשים כאלה להפרט והכלל יהיו ממש ההיפך מה שגעגועים אלהיים\n\nМЕТАЯЗЫК\nТолкование можно составлять только из следующих 65 семантических примитивов и простых грамматических связей между ними:\n\nСубстантивы: Я, ТЫ, КТО-ТО, ЧТО-ТО / ВЕЩЬ, ЛЮДИ, ТЕЛО\nРеляционные субстантивы: РОД / ВИД, ЧАСТЬ\nДетерминаторы: ЭТОТ, ТОТ ЖЕ, ДРУГОЙ\nКванторы: ОДИН, ДВА, НЕКОТОРЫЕ, ВСЕ, МНОГО, МАЛО\nОценки: ХОРОШИЙ, ПЛОХОЙ\nДескрипторы: БОЛЬШОЙ, МАЛЕНЬКИЙ\nМентальные предикаты: ДУМАТЬ, ЗНАТЬ, ХОТЕТЬ, НЕ ХОТЕТЬ, ЧУВСТВОВАТЬ, ВИДЕТЬ, СЛЫШАТЬ\nРечь: СКАЗАТЬ, СЛОВА, ПРАВДА\nДействия, события, движение: ДЕЛАТЬ, ПРОИСХОДИТЬ, ДВИГАТЬСЯ\nСуществование, местонахождение, обладание, спецификация: БЫТЬ (где-то), ЕСТЬ, БЫТЬ (кем-то / чем-то), (у кого-то) ЕСТЬ\nЖизнь и смерть: ЖИТЬ, УМЕРЕТЬ\nВремя: КОГДА / ВРЕМЯ, СЕЙЧАС, ДО, ПОСЛЕ, ДОЛГО, НЕДОЛГО, НЕКОТОРОЕ ВРЕМЯ, МОМЕНТ\nПространство: ГДЕ / МЕСТО, ЗДЕСЬ, НАД, ПОД, ДАЛЕКО, БЛИЗКО, СТОРОНА, ВНУТРИ, КАСАТЬСЯ\nЛогические понятия: НЕ, МОЖЕТ БЫТЬ, МОЧЬ, ПОТОМУ ЧТО, ЕСЛИ\nИнтенсификатор, аугментатор: ОЧЕНЬ, БОЛЬШЕ\nСходство: КАК / ТАК\n\nПРАВИЛА\n1. Используй только перечисленные примитивы. Не используй сложные слова (например, «желание», «отсутствие», «стремление», «надежда»), даже если они кажутся простыми. Само слово כמיהה, его переводы и синонимы в толковании использовать нельзя.\n2. Толкование эмоционального понятия строится как прототипический сценарий: «кто-то чувствует что-то; иногда человек думает так: „…“; из-за этого этот человек чувствует что-то …; так, как чувствуют люди, когда думают так».\n3. Каждая строка толкования — одна короткая простая фраза.\n4. Толкование должно быть таким, чтобы его можно было дословно перевести на любой язык.\n\nШАГИ\n1. Прочитай все контексты и определи, в каких сферах жизни употребляется слово.\n2. Найди то, что общее для всех этих сфер.\n3. Составь толкование из примитивов.\n4. Проверь каждое слово толкования по списку примитивов и замени всё, чего в списке нет.\n\nФОРМАТ ОТВЕТА\nВерни только JSON, без пояснений до и после:\n{\"spheres\": [\"сфера употребления 1\", \"сфера употребления 2\"], \"explication\": [\"строка 1\", \"строка 2\"]}\nСферы и толкование пиши на русском языке.\n",
 "en": "You are a specialist in lexical semantics working within the Natural Semantic Metalanguage (NSM) theory of A. Wierzbicka and C. Goddard.\n\nTASK\nWrite a semantic explication of the Hebrew noun כמיהה in the Natural Semantic Metalanguage.\n\nMATERIAL\nBelow are 24 real uses of this word from a corpus of Modern Hebrew. Study them carefully: the explication must describe the meaning shared by all of these uses.\n\n1. הרחבה. למעשה, אם לא מאחורי כל פנייה בין אישית, לפחות מאחורי כל פנייה לטיפול מסתתרים צרכים ראשוניים, מסתתרת כמיהה ראשונית להיענות אימהית, ותפקידו של המטפל הוא לזהות אותה. עדיין היענות לצרכים אלה אינה פשוטה והיא דורשת סוג של אומנות,\n2. מספור, דרך מאות שנים של סבל וגלות, מעשי טבח ואף רצח עם. אך למרות כל זאת, העם היהודי שמר על זהותו הייחודית ועל מסורותיו וכן על כמיהתו הארוכה לשוב למולדת.' כך דיבר אובמה על זכות הישראלים על אדמת פלסטין... אתם מתארים לעצמכם את הנשיא האמריקאי מקריא את הקטע ה\n3. סופת טורנדו. בעקבות זעמה מתעוררת בדורותי המשאלה הילדית לעזוב את ההורים \"הרעים\" ולהתחיל מחדש. הביקורת על ההורים, כמו הכמיהה לעזוב אותם, אינה מודעת אצל ילדים צעירים. החשיבה הביקורתית נתפסת כבוגדנית ומעוררת אשמה, בושה וחרדה ולכן היא מודחקת אל מ\n4. של מדינת ישראל, אלא בקוממיותו של האזרח הערבי הנרמס על ידי מדינת הלאום המדירה, הדכאנית והמושחתת שלו. אולם יש במאמר געגוע, כמיהה, למילה שרבים בשמאל הרדיקלי (בארץ ובחו\"ל) שכחו, או היו מעדיפים לחיות בלעדיו: מדינה. אלה שחיים בתוך מדינה חזקה, יציבה,\n5. והנשימות שלי מואצות. האם אי-פעם יגיע הרגע שבו אהיה לידו בלי להשתוקק אליו? בלי לייחל למגע עורו כנגד עורי? זאת אפילו לא כמיהה מינית – לפחות לא לגמרי. זה צורך. כאילו ההישרדות שלי תלויה בו. כאילו אנחנו שני חצאים של שלם אחד ואף אחד מאיתנו לא יכול לשרוד\n6. שסקס הוא הכוח החזק ביותר שמאפשר את מערכת היחסים. הם טועים. אהבה רומנטית, להבדיל מערנות מינית, היא למעשה חזקה יותר. הכמיהה לאהבה רומנטית יכולה להיות חזקה יותר מהרצון לחיות, וחזקה הרבה יותר מכל דחף מיני שהוא. הלן פישר, אנתרופולוגית אמריקאית,\n7. שיטת סאטיה כל אדם שואף להרמוניה, רווחה אישית ושקט נפשי. רצונו של האדם על מכלול מטרותיו, שאיפותיו וחלומותיו, מסתכם בכמיהתו לשקט, לתכלית וליושר פנימי המתקיימים בהלימה רגשית נפשית ופיסית בחייו. מה שנוכח כאן ועכשיו – התייחסותו של האדם להווה\n8. בדרכך ומספרת שאת לא צריכה לתת דין וחשבון כי לאף אחד מהצדדים אין באמת ציפיות. אז שימי לב איך מתעוררים אצלך רגשות נוספים כי הכמיהה לקרבה אמיתית לא מתכהה גם לנוכח תשוקה מינית עזה. תמיד תזכרי את הבוקר שאחרי וגם אם זה אומר שיש מקרים שתצאי מחדר השינה בלי\n9. וזלדה. ואלה רק שלושה. נסים אלוני אבֵל על מגע ידו המשחיר. שאול טשרניחובסקי אבֵל על אי-המגע. השושנה שלו הופכת לכמיהה בלתי מושגת ומתמדת, לתשוקה המאוכלת כבר למן ההתחלה. בדומה לאלוני, יש בפתיחת השיר מעין הבטחת-אושר שוות-כל, ברכת אל שורה מ\n10. . רוב הספרות הנוגעת בנושא, אם כן, רואה בחלומות שבהם מופיע המטפל מפורשות (ללא מסווה) כמשקפים בעיות קשות אצל המטופל )למשל, כמיהה עזה להתמזגות או פגיעה ביכולת הסימבולית) או אצל המטפל ובקשר הטיפולי )(למשל, קשיים הקשורים להעברה נגדית, חדירת המציאות,\n11. משנה תורה. במשנה תורה, לעומת זאת, פסגת אהבת ה', אשר יסודה היא בשבח ובהתפעמות מחכמתו וגדולתו המשתקפים בבריאה, היא צמאון וכמיהה להכרה נוספת. צמאון זה מתואר בהלכות תשובה (ראו בנספח; וגם קודם לכן שם, בהלכה ג) כשגיון וכאובססיה כאילו חולנית. כך, כנראה,\n12. עת שהמרחק רב בין הגופות מתנווט אני אלייך במחשבות כמדריך את התקוות מגעגועים אל משימות, מחלום אל הוויות, מציפיה אל עשייה, מכמיהה אל נגיעה, מאהבה אל תאווה. חכי לי עד אחזור ואלייך אחדור, כי נמאס מהחידלון שהמרחק מאלצו. במעגלי צדק תלפיות, יד חרוצים 19\n13. \", מולו. בתהליך זה תוך השתקפות באחר העומד מנגד מתפתח העצמי ובוקעת התודעה. (consciousness) בשלב מאוחר יותר בחיים כאשר הכמיהה להידבקות תהיה משוחררת מן הצורך הקמאי להשלים עצמי באמצעות החיבור, יוכל להתקיים סוף סוף החיבור והדבקות הפעם במישור ה\n14. קהילת יוצאי אתיופיה, חוגגים את חג הסיגד המרכז הרפואי ברזילי מברך את בני קהילת יוצאי אתיופיה, לרגל חג הסיגד. החג מסמל את הכמיהה לעלייה לארץ, ואת חידוש הברית של יהודי אתיופיה עם עמם ועם תורתם, והוא נחוג בכל שנה בכ\"ט בחשוון, 50 יום לאחר יום הכיפורים. ב\n15. של הצבא כגואל העם ומגן העצמאות, הם חונכו על מיתוס הפוך – הצבא כמרכז של כפירה, כמרד נגד השם, כמקום זימה ותועבה. אין לנו שום כמיהה להגיע למוסד הזה. שלישית – עזבנו מסגרת נוקשה עם כללים שאנו כופרים בהם. הצבא הוא מבחינות רבות מסגרת דומה. צריך להאמין ב\n16. באמת ובתמים צועדים בנתיבו של פטרוס. אלוהים שיבח את פטרוס על אמונת האמת ואהבת האמת שלו ועל נאמנותו לאלוהים. ובזכות כנותו וכמיהתו לאלוהים בלבו, אלוהים עשה אותו למושלם. אם אכן יש לכם אותה אהבה ואותה אמונה שהייתה לפטרוס, הרי שישוע לבטח יעשה אתכם ל\n17. , וניתן לאחד רוח ברוח. זוהי איננה כמיהה לבלתי מושג אלא כמיהה למושג. כאשר מתעוררת אצל האדם המשיכה המינית עליו לדעת שזוהי הכמיהה לבלתי מושג. אין לו אפשרות להגיע לרמת חיבור כזאת בעולם הזה. האשליה כאילו ... לחבר בין אהבה למשיכה מינית, בין מהות לצורה.\n18. שם ממשפחות חרדיות לעילא ולעילא, והוא רואה לעיתים תופעות מאכזבות, בהתחלה תמימותו נסדקת, ובהמשך הוא מושפע בעצמו... וה כמיהה הכנה שלו ל\"שבתי\" \"כל ימי חיי! \", ולדבקות מוחלטת בתורה ובנותנה, ולקנין גדלות של ממש ולירא\"ש ללא כחל ושרק, לפעמים, הבנין ה\n19. פרטני וזוגי בתחום הטיפול התומך בטיפולי פוריות, מסבירה על חווית ההתמודדות עם הטיפולים. \"הבה לי בנים, ואם אין מתה אנכי\": הכמיהה לילד והשפעתם של טיפולי הפוריות \"הבה לי בנים, ואם אין מתה אנכי\" (בראשית ל'), אומרת רחל המקראית ביאושה, אחרי שנים של עקרות.\n20. חלק מהשליחים והמגויסים טענו כי סמכות ההגנה אכן התקבלה בהתלהבות וללא עוררין. שמואל ארנר, שהתגייס להגנה בגרמניה, מתאר את כמיהתו ונכונות הנוער להצטרף לארגון ההגנה מתוך מטרה אחת: עלייה לארץ ישראל, ומתוך כך קיבלו רבים את סמכות ההגנה. ככל שעבר הזמן, ו\n21. בכך הזדמנות. הייתי אז חבר-כנסת. אינני אוהב, בלשון המעטה, את ההימנון הלאומי שלנו. המילים נכתבו לפני מאה שנה וביטאו את כמיהת היהודים בגולה לארץ-ישראל. גם אחרי ששונו המילים של הפזמון (''עוד לא אבדה תקוותנו / התקווה הנושנה / לשוב לארץ אבותינו / ה\n22. ולשכנע את האדם שהיא מרוקנת את חייו מכל משמעות והופכת אותם לחלולים. בשלב שני, צריך לעורר את הצד העליון שבאדם, להעיר את כמיהתו לרוחניות, לערכים, לאידיאלים, להציב בפניו אתגרים רוחניים, וממילא ישוב גם אל האמונה ואל הקדושה. לעומת זאת, את תרבות ה\n23. ערי רקיע ויגיעו עד לפני כיסא הכבוד. \"יותר גוף יותר כתיבה\" הלן סיסקו, \"צחוקה של המדוזה\" בשירה האמונית קיימת באופן אימננטי כמיהה וחתירה לקראת מפגש בלתי מתווך עם האל. בדברים שלהלן אבקש לבדוק דרכי התמודדות שונות במפגש עם האל, המוצעות בשיריהן של שלוש\n24. האלהיים אם לא יפותחו כראוי בחכמה האצילית ובחכמת החיים, בגבורת הנשמה ובגבורת הגוף, יכולים הם לפול ממדרגתם לידי איזה כמיהה חולנית, שינטל מהם כל זיום ותפארתם, עד אשר מה שיגרמו געגועים נחלשים כאלה להפרט והכלל יהיו ממש ההיפך מה שגעגועים אלהיים\n\nMETALANGUAGE\nThe explication may be built only from the following 65 semantic primes and simple grammatical links between them:\n\nSubstantives: I, YOU, SOMEONE, SOMETHING / THING, PEOPLE, BODY\nRelational substantives: KIND, PART\nDeterminers: THIS, THE SAME, OTHER\nQuantifiers: ONE, TWO, SOME, ALL, MUCH / MANY, LITTLE / FEW\nEvaluators: GOOD, BAD\nDescriptors: BIG, SMALL\nMental predicates: THINK, KNOW, WANT, DON'T WANT, FEEL, SEE, HEAR\nSpeech: SAY, WORDS, TRUE\nActions, events, movement: DO, HAPPEN, MOVE\nExistence, location, possession, specification: BE (SOMEWHERE), THERE IS, BE (SOMEONE / SOMETHING), (IS) MINE\nLife and death: LIVE, DIE\nTime: WHEN / TIME, NOW, BEFORE, AFTER, A LONG TIME, A SHORT TIME, FOR SOME TIME, MOMENT\nSpace: WHERE / PLACE, HERE, ABOVE, BELOW, FAR, NEAR, SIDE, INSIDE, TOUCH\nLogical concepts: NOT, MAYBE, CAN, BECAUSE, IF\nIntensifier, augmentor: VERY, MORE\nSimilarity: LIKE / AS / WAY\n\nRULES\n1. Use only the primes listed above. Do not use complex words (for example, \"desire\", \"absence\", \"striving\", \"hope\"), even if they seem simple. Do not use the word כמיהה itself, its translations or its synonyms in the explication.\n2. An explication of an emotion concept is built as a prototypical scenario: \"someone feels something; sometimes a person thinks like this: '…'; because of this, this person feels something …; like people feel when they think like this\".\n3. Each line of the explication is one short, simple phrase.\n4. The explication must be such that it can be translated word for word into any language.\n\nSTEPS\n1. Read all the contexts and determine in which spheres of life the word is used.\n2. Find what is common to all of these spheres.\n3. Build the explication from the primes.\n4. Check every word of the explication against the list of primes and replace anything that is not on the list.\n\nRESPONSE FORMAT\nReturn only JSON, with no explanations before or after it:\n{\"spheres\": [\"sphere of use 1\", \"sphere of use 2\"], \"explication\": [\"line 1\", \"line 2\"]}\nWrite the spheres and the explication in English.\n",
 "he": "אתה מומחה לסמנטיקה לקסיקלית הפועל במסגרת תאוריית המטא-שפה הסמנטית הטבעית (Natural Semantic Metalanguage, NSM) של א' וייז'ביצקה וק' גודארד.\n\nהמשימה\nחבר הגדרה סמנטית (אקספליקציה) של שם העצם העברי כמיהה במטא-שפה הסמנטית הטבעית.\n\nהחומר\nלהלן 24 היקרויות אמיתיות של המילה מתוך קורפוס של העברית בת זמננו. עיין בהן היטב: ההגדרה צריכה לתאר את המשמעות המשותפת לכל ההיקרויות האלה.\n\n1. הרחבה. למעשה, אם לא מאחורי כל פנייה בין אישית, לפחות מאחורי כל פנייה לטיפול מסתתרים צרכים ראשוניים, מסתתרת כמיהה ראשונית להיענות אימהית, ותפקידו של המטפל הוא לזהות אותה. עדיין היענות לצרכים אלה אינה פשוטה והיא דורשת סוג של אומנות,\n2. מספור, דרך מאות שנים של סבל וגלות, מעשי טבח ואף רצח עם. אך למרות כל זאת, העם היהודי שמר על זהותו הייחודית ועל מסורותיו וכן על כמיהתו הארוכה לשוב למולדת.' כך דיבר אובמה על זכות הישראלים על אדמת פלסטין... אתם מתארים לעצמכם את הנשיא האמריקאי מקריא את הקטע ה\n3. סופת טורנדו. בעקבות זעמה מתעוררת בדורותי המשאלה הילדית לעזוב את ההורים \"הרעים\" ולהתחיל מחדש. הביקורת על ההורים, כמו הכמיהה לעזוב אותם, אינה מודעת אצל ילדים צעירים. החשיבה הביקורתית נתפסת כבוגדנית ומעוררת אשמה, בושה וחרדה ולכן היא מודחקת אל מ\n4. של מדינת ישראל, אלא בקוממיותו של האזרח הערבי הנרמס על ידי מדינת הלאום המדירה, הדכאנית והמושחתת שלו. אולם יש במאמר געגוע, כמיהה, למילה שרבים בשמאל הרדיקלי (בארץ ובחו\"ל) שכחו, או היו מעדיפים לחיות בלעדיו: מדינה. אלה שחיים בתוך מדינה חזקה, יציבה,\n5. והנשימות שלי מואצות. האם אי-פעם יגיע הרגע שבו אהיה לידו בלי להשתוקק אליו? בלי לייחל למגע עורו כנגד עורי? זאת אפילו לא כמיהה מינית – לפחות לא לגמרי. זה צורך. כאילו ההישרדות שלי תלויה בו. כאילו אנחנו שני חצאים של שלם אחד ואף אחד מאיתנו לא יכול לשרוד\n6. שסקס הוא הכוח החזק ביותר שמאפשר את מערכת היחסים. הם טועים. אהבה רומנטית, להבדיל מערנות מינית, היא למעשה חזקה יותר. הכמיהה לאהבה רומנטית יכולה להיות חזקה יותר מהרצון לחיות, וחזקה הרבה יותר מכל דחף מיני שהוא. הלן פישר, אנתרופולוגית אמריקאית,\n7. שיטת סאטיה כל אדם שואף להרמוניה, רווחה אישית ושקט נפשי. רצונו של האדם על מכלול מטרותיו, שאיפותיו וחלומותיו, מסתכם בכמיהתו לשקט, לתכלית וליושר פנימי המתקיימים בהלימה רגשית נפשית ופיסית בחייו. מה שנוכח כאן ועכשיו – התייחסותו של האדם להווה\n8. בדרכך ומספרת שאת לא צריכה לתת דין וחשבון כי לאף אחד מהצדדים אין באמת ציפיות. אז שימי לב איך מתעוררים אצלך רגשות נוספים כי הכמיהה לקרבה אמיתית לא מתכהה גם לנוכח תשוקה מינית עזה. תמיד תזכרי את הבוקר שאחרי וגם אם זה אומר שיש מקרים שתצאי מחדר השינה בלי\n9. וזלדה. ואלה רק שלושה. נסים אלוני אבֵל על מגע ידו המשחיר. שאול טשרניחובסקי אבֵל על אי-המגע. השושנה שלו הופכת לכמיהה בלתי מושגת ומתמדת, לתשוקה המאוכלת כבר למן ההתחלה. בדומה לאלוני, יש בפתיחת השיר מעין הבטחת-אושר שוות-כל, ברכת אל שורה מ\n10. . רוב הספרות הנוגעת בנושא, אם כן, רואה בחלומות שבהם מופיע המטפל מפורשות (ללא מסווה) כמשקפים בעיות קשות אצל המטופל )למשל, כמיהה עזה להתמזגות או פגיעה ביכולת הסימבולית) או אצל המטפל ובקשר הטיפולי )(למשל, קשיים הקשורים להעברה נגדית, חדירת המציאות,\n11. משנה תורה. במשנה תורה, לעומת זאת, פסגת אהבת ה', אשר יסודה היא בשבח ובהתפעמות מחכמתו וגדולתו המשתקפים בבריאה, היא צמאון וכמיהה להכרה נוספת. צמאון זה מתואר בהלכות תשובה (ראו בנספח; וגם קודם לכן שם, בהלכה ג) כשגיון וכאובססיה כאילו חולנית. כך, כנראה,\n12. עת שהמרחק רב בין הגופות מתנווט אני אלייך במחשבות כמדריך את התקוות מגעגועים אל משימות, מחלום אל הוויות, מציפיה אל עשייה, מכמיהה אל נגיעה, מאהבה אל תאווה. חכי לי עד אחזור ואלייך אחדור, כי נמאס מהחידלון שהמרחק מאלצו. במעגלי צדק תלפיות, יד חרוצים 19\n13. \", מולו. בתהליך זה תוך השתקפות באחר העומד מנגד מתפתח העצמי ובוקעת התודעה. (consciousness) בשלב מאוחר יותר בחיים כאשר הכמיהה להידבקות תהיה משוחררת מן הצורך הקמאי להשלים עצמי באמצעות החיבור, יוכל להתקיים סוף סוף החיבור והדבקות הפעם במישור ה\n14. קהילת יוצאי אתיופיה, חוגגים את חג הסיגד המרכז הרפואי ברזילי מברך את בני קהילת יוצאי אתיופיה, לרגל חג הסיגד. החג מסמל את הכמיהה לעלייה לארץ, ואת חידוש הברית של יהודי אתיופיה עם עמם ועם תורתם, והוא נחוג בכל שנה בכ\"ט בחשוון, 50 יום לאחר יום הכיפורים. ב\n15. של הצבא כגואל העם ומגן העצמאות, הם חונכו על מיתוס הפוך – הצבא כמרכז של כפירה, כמרד נגד השם, כמקום זימה ותועבה. אין לנו שום כמיהה להגיע למוסד הזה. שלישית – עזבנו מסגרת נוקשה עם כללים שאנו כופרים בהם. הצבא הוא מבחינות רבות מסגרת דומה. צריך להאמין ב\n16. באמת ובתמים צועדים בנתיבו של פטרוס. אלוהים שיבח את פטרוס על אמונת האמת ואהבת האמת שלו ועל נאמנותו לאלוהים. ובזכות כנותו וכמיהתו לאלוהים בלבו, אלוהים עשה אותו למושלם. אם אכן יש לכם אותה אהבה ואותה אמונה שהייתה לפטרוס, הרי שישוע לבטח יעשה אתכם ל\n17. , וניתן לאחד רוח ברוח. זוהי איננה כמיהה לבלתי מושג אלא כמיהה למושג. כאשר מתעוררת אצל האדם המשיכה המינית עליו לדעת שזוהי הכמיהה לבלתי מושג. אין לו אפשרות להגיע לרמת חיבור כזאת בעולם הזה. האשליה כאילו ... לחבר בין אהבה למשיכה מינית, בין מהות לצורה.\n18. שם ממשפחות חרדיות לעילא ולעילא, והוא רואה לעיתים תופעות מאכזבות, בהתחלה תמימותו נסדקת, ובהמשך הוא מושפע בעצמו... וה כמיהה הכנה שלו ל\"שבתי\" \"כל ימי חיי! \", ולדבקות מוחלטת בתורה ובנותנה, ולקנין גדלות של ממש ולירא\"ש ללא כחל ושרק, לפעמים, הבנין ה\n19. פרטני וזוגי בתחום הטיפול התומך בטיפולי פוריות, מסבירה על חווית ההתמודדות עם הטיפולים. \"הבה לי בנים, ואם אין מתה אנכי\": הכמיהה לילד והשפעתם של טיפולי הפוריות \"הבה לי בנים, ואם אין מתה אנכי\" (בראשית ל'), אומרת רחל המקראית ביאושה, אחרי שנים של עקרות.\n20. חלק מהשליחים והמגויסים טענו כי סמכות ההגנה אכן התקבלה בהתלהבות וללא עוררין. שמואל ארנר, שהתגייס להגנה בגרמניה, מתאר את כמיהתו ונכונות הנוער להצטרף לארגון ההגנה מתוך מטרה אחת: עלייה לארץ ישראל, ומתוך כך קיבלו רבים את סמכות ההגנה. ככל שעבר הזמן, ו\n21. בכך הזדמנות. הייתי אז חבר-כנסת. אינני אוהב, בלשון המעטה, את ההימנון הלאומי שלנו. המילים נכתבו לפני מאה שנה וביטאו את כמיהת היהודים בגולה לארץ-ישראל. גם אחרי ששונו המילים של הפזמון (''עוד לא אבדה תקוותנו / התקווה הנושנה / לשוב לארץ אבותינו / ה\n22. ולשכנע את האדם שהיא מרוקנת את חייו מכל משמעות והופכת אותם לחלולים. בשלב שני, צריך לעורר את הצד העליון שבאדם, להעיר את כמיהתו לרוחניות, לערכים, לאידיאלים, להציב בפניו אתגרים רוחניים, וממילא ישוב גם אל האמונה ואל הקדושה. לעומת זאת, את תרבות ה\n23. ערי רקיע ויגיעו עד לפני כיסא הכבוד. \"יותר גוף יותר כתיבה\" הלן סיסקו, \"צחוקה של המדוזה\" בשירה האמונית קיימת באופן אימננטי כמיהה וחתירה לקראת מפגש בלתי מתווך עם האל. בדברים שלהלן אבקש לבדוק דרכי התמודדות שונות במפגש עם האל, המוצעות בשיריהן של שלוש\n24. האלהיים אם לא יפותחו כראוי בחכמה האצילית ובחכמת החיים, בגבורת הנשמה ובגבורת הגוף, יכולים הם לפול ממדרגתם לידי איזה כמיהה חולנית, שינטל מהם כל זיום ותפארתם, עד אשר מה שיגרמו געגועים נחלשים כאלה להפרט והכלל יהיו ממש ההיפך מה שגעגועים אלהיים\n\nהמטא-שפה\nמותר לחבר את ההגדרה רק מתוך 65 הפרימיטיבים הסמנטיים הבאים ומקשרים דקדוקיים פשוטים ביניהם:\n\nשמות עצם: אני, אתה / את, מישהו, משהו, אנשים, גוף\nשמות עצם יחסיים: סוג, חלק\nמיידעים: הזה / הזאת, אותו הדבר, אחר\nמכמתים: אחד, שניים, חלק מ־, הכול / כל, הרבה, מעט\nהערכה: טוב, רע\nתיאור: גדול, קטן\nפרדיקטים מנטליים: חשב, ידע, רצה, לא רצה, הרגיש, ראה, שמע\nדיבור: אמר, מילים, נכון\nפעולות, אירועים, תנועה: עשה, קרה, זז / נע\nקיום, מיקום, שייכות, ציון: נמצא (במקום כלשהו), יש, הוא (להיות), שלי\nחיים ומוות: חי, מת\nזמן: מתי / זמן, עכשיו, לפני, אחרי, הרבה זמן, זמן קצר, למשך זמן מה, רגע\nמרחב: איפה / מקום, פה / כאן, מעל, מתחת, רחוק, קרוב, צד, בתוך, נגע\nמושגים לוגיים: לא, אולי, יכול, בגלל, אם\nמעצים, מגביר: מאוד, יותר\nדמיון: כמו\n\nכללים\n1. השתמש רק בפרימיטיבים שברשימה. אל תשתמש במילים מורכבות (למשל \"רצון\", \"היעדר\", \"שאיפה\", \"תקווה\"), גם אם הן נראות פשוטות. אסור להשתמש בהגדרה במילה כמיהה עצמה, בתרגומיה או במילים נרדפות לה.\n2. הגדרה של מושג רגשי בנויה כתסריט אב-טיפוסי: \"מישהו מרגיש משהו; לפעמים אדם חושב כך: '…'; בגלל זה האדם הזה מרגיש משהו …; כמו שאנשים מרגישים כשהם חושבים כך\".\n3. כל שורה בהגדרה היא משפט אחד קצר ופשוט.\n4. ההגדרה צריכה להיות כזו שאפשר לתרגם אותה מילה במילה לכל שפה.\n\nשלבים\n1. קרא את כל ההקשרים וקבע באילו תחומי חיים משמשת המילה.\n2. מצא מה משותף לכל התחומים האלה.\n3. חבר את ההגדרה מתוך הפרימיטיבים.\n4. בדוק כל מילה בהגדרה מול רשימת הפרימיטיבים והחלף כל מה שאינו מופיע ברשימה.\n\nמבנה התשובה\nהחזר JSON בלבד, ללא הסברים לפניו או אחריו:\n{\"spheres\": [\"תחום שימוש 1\", \"תחום שימוש 2\"], \"explication\": [\"שורה 1\", \"שורה 2\"]}\nכתוב את התחומים ואת ההגדרה בעברית.\n"
}

## 4. Функции запроса

In [ ]:
def call_openrouter(model_id, prompt):
    assert model_id.endswith(":free"), "Разрешены только бесплатные модели (:free)"
    r = requests.post("https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENROUTER_KEY}"},
        json={"model": model_id, "temperature": TEMPERATURE, "max_tokens": MAX_TOKENS,
              "messages": [{"role": "user", "content": prompt}]}, timeout=300)
    r.raise_for_status()
    content = (r.json()["choices"][0]["message"].get("content") or "").strip()
    if not content:
        raise ValueError("пустой ответ модели")
    return content

def call(model, prompt):
    for t in range(TRIES):
        try:
            return call_openrouter(model["id"], prompt)
        except Exception as e:
            print("  ошибка:", e, "| повтор через", 15 * (t + 1), "с")
            time.sleep(15 * (t + 1))
    return None          # отказ: засчитывается как 0 баллов

def parse(text):
    """Достаёт JSON {spheres, explication} из ответа (убирает <think> и ```json)."""
    if not text: return None
    t = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    t = re.sub(r"```(json)?", "", t)
    a, b = t.find("{"), t.rfind("}")
    try:
        return json.loads(t[a:b + 1])
    except Exception:
        return None

## 5. Прогон (30 запросов, около часа)
Ответы сохраняются в `results.json` после каждого запроса; при перезапуске уже полученные ответы не запрашиваются заново.

In [ ]:
res = json.load(open(OUT)) if os.path.exists(OUT) else []
done = {(r["model"], r["lang"], r["iter"]) for r in res if r["raw"]}
for it in range(1, N_ITER + 1):
    for m in MODELS:
        for lang in LANGS:
            if (m["name"], lang, it) in done:
                continue
            print(m["name"], lang, it)
            raw = call(m, PROMPTS[lang])
            res = [r for r in res if (r["model"], r["lang"], r["iter"]) != (m["name"], lang, it)]
            res.append({"model": m["name"], "model_id": m["id"], "lang": lang, "iter": it,
                        "temperature": TEMPERATURE, "raw": raw, "parsed": parse(raw)})
            json.dump(res, open(OUT, "w"), ensure_ascii=False, indent=1)
            time.sleep(PAUSE)
print("ответов:", sum(1 for r in res if r["raw"]), "из", len(MODELS) * len(LANGS) * N_ITER)
from google.colab import files
files.download(OUT)

## 6. Таблица для слепой оценки
Строки перемешаны, модель скрыта за кодом T01–T30. Ключ «код → модель» сохраняется отдельно.

In [ ]:
random.seed(7)
order = list(range(len(res))); random.shuffle(order)
rows, key = [], []
for i, idx in enumerate(order):
    r = res[idx]; p = r["parsed"] or {}; code = f"T{i+1:02d}"
    key.append({"code": code, "model": r["model"], "lang": r["lang"], "iter": r["iter"]})
    rows.append({"code": code, "lang": r["lang"],
                 "spheres": "; ".join(p.get("spheres", [])) if r["raw"] else "",
                 "explication": "\n".join(p.get("explication", [])) if r["raw"] else "— ОТКАЗ —",
                 "K1": "", "K2": "", "K3": "", "K4": "", "K5": "", "comment": ""})
pd.DataFrame(rows).to_excel("scoring.xlsx", index=False)
pd.DataFrame(key).to_csv("key.csv", index=False)
files.download("scoring.xlsx"); files.download("key.csv")

## 7. Критерий К1: автоматическая сверка с метаязыком
Каждое слово толкования сверяется со списком 65 примитивов (с учётом словоформ — лемматизатор pymorphy3 для русского) и служебных слов. Выводятся слова вне списка; грамматика примитивов («у меня не есть», «אני רצה») проверяется вручную.

Правило: 0 слов вне списка → 2; 1–2 слова или ошибки грамматики → 1; ≥ 3 слов, другой язык или смесь языков → 0.

In [ ]:
!pip -q install pymorphy3 pymorphy3-dicts-ru

In [ ]:
import json, re, pymorphy3
m = pymorphy3.MorphAnalyzer()
res = json.load(open(OUT))

# --- allowed vocab: primes (+allolexes) + grammatical links/pronouns ---
RU = set("""я ты кто-то что-то вещь люди тело род вид часть этот тот же самый другой один два некоторый весь много мало
хороший плохой большой маленький думать знать хотеть чувствовать видеть слышать сказать говорить слово правда
делать происходить случиться двигаться быть есть жить умереть когда время сейчас до после долго недолго некоторое момент
где место здесь над под далеко близко сторона внутри касаться не может мочь потому что если очень больше как так
человек кто что это у в на о об с и а чтобы такой он она они его её их ему ей им меня мне мой свой себя тот это
иногда некоторое когда-то тогда там нет""".split())
EN = set("""i you someone something thing people body kind part this the same other else one two some all much many little few
good bad big small think know want feel see hear say word words true do happen move be is are was were am there mine
live die when time now before after long short moment where place here above below far near side inside touch not no
maybe can because if very more like as way person a an it its they them their he she his her me my of to for at in on
about with that what so how these those someone's times have has thing's""".split())
HE = set("""אני אתה את מישהו משהו אנשים גוף סוג חלק זה הזה הזאת זאת אותו הדבר אחר אחד שניים כל הכול הרבה מעט טוב רע גדול קטן
חשב חושב חושבים ידע יודע רצה רוצה רוצים הרגיש מרגיש מרגישים ראה רואה שמע שומע אמר אומר מילים נכון עשה עושה קרה קורה זז נע
נגע נוגע חי מת מתי זמן עכשיו לפני אחרי קצר למשך מה רגע איפה מקום פה כאן מעל מתחת רחוק קרוב צד בתוך לא אולי יכול בגלל אם מאוד יותר כמו
יש שלי הוא היא הם להיות נמצא כך כמה ש של לו לה""".split())
PREF = "השוכבלמ"

def tokens(s):
    return re.findall(r"[A-Za-z']+|[֐-׿]+|[а-яёА-ЯЁ-]+|[一-鿿]+", s)

def bad_ru(t):
    w = t.lower().strip('-')
    if not w: return False
    if w in RU: return False
    return m.parse(w)[0].normal_form not in RU
def bad_en(t):
    w = t.lower().strip("'")
    for cand in [w, w[:-1] if w.endswith('s') else w, w[:-2] if w.endswith('es') else w, w[:-1] if w.endswith('d') else w]:
        if cand in EN: return False
    return True
def bad_he(t):
    if t in HE: return False
    for i in range(1,3):
        if t[i:] in HE and all(c in PREF for c in t[:i]): return False
    return True

def check(lang, lines):
    out = []
    for line in lines:
        for t in tokens(line):
            if re.match(r"[一-鿿]", t): out.append(t); continue
            if re.match(r"[A-Za-z]", t):
                if lang=='en': 
                    if bad_en(t): out.append(t)
                else: out.append(t)          # латиница в ru/he = чужой язык
            elif re.match(r"[֐-׿]", t):
                if lang!='he' or bad_he(t): out.append(t)
            else:
                if lang!='ru' or bad_ru(t): out.append(t)
    return out

rows=[]
for r in res:
    ex = (r['parsed'] or {}).get('explication', []) if r['raw'] else []
    bad = check(r['lang'], ex) if ex else None
    rows.append(dict(model=r['model'],lang=r['lang'],iter=r['iter'],n_lines=len(ex),bad=bad))
for x in rows:
    b = x['bad']
    print(x['model'],x['lang'],x['iter'], 'ОТКАЗ' if b is None else f"{len(set(b))} | {sorted(set(b))[:25]}")


## 8. Статистика по экспертной оценке
Загрузите заполненный `scoring.xlsx` (колонки K1–K5) и `key.csv` в Colab. Для каждой пары «модель × язык» считаются среднее и выборочное стандартное отклонение итогового балла (0–10).

In [ ]:
sc = pd.read_excel("scoring.xlsx").merge(pd.read_csv("key.csv"), on="code")
sc["total"] = sc[["K1", "K2", "K3", "K4", "K5"]].sum(axis=1)
print(sc.groupby(["model", "lang"])["total"].agg(["mean", "std"]).round(2))
print(sc.groupby("lang")["total"].agg(["mean", "std"]).round(2))
print(sc.groupby("model")["total"].agg(["mean", "std"]).round(2))
print(sc.groupby(["model", "lang"])[["K1", "K2", "K3", "K4", "K5"]].mean().round(2))

In [ ]:
import matplotlib.pyplot as plt, numpy as np
g = sc.groupby(["model", "lang"])["total"].agg(["mean", "std"])
fig, ax = plt.subplots(figsize=(8, 4.5)); x = np.arange(3); w = 0.36
for i, (m, c) in enumerate([("Nemotron3", "#2a78d6"), ("Dots3", "#eb6834")]):
    mu = [g.loc[(m, l), "mean"] for l in LANGS]; sd = [g.loc[(m, l), "std"] for l in LANGS]
    ax.bar(x + (i - .5) * w, mu, w, yerr=sd, capsize=4, color=c, label=m)
ax.set_xticks(x, ["русский", "английский", "иврит"]); ax.set_ylim(0, 10)
ax.set_ylabel("средний балл (0–10)"); ax.legend(frameon=False)
ax.set_title("Среднее ± стандартное отклонение по 5 итерациям"); plt.show()